---
title: "FHIRScan | DIZ-Vergleich"
subtitle: "Unterschiede und Gemeinsamkeiten der FHIR-Daten in den Datenintegrationszentren"
lang: de
format:
  html:
    theme: cosmo
    css: assets/bhc-report.css
    toc: true
    toc-depth: 2
    toc-title: "Inhalt"
    toc-location: left
    number-sections: true
    embed-resources: true
    page-layout: full
execute:
  echo: false
  warning: false
jupyter: python3
---

In [ ]:
#| include: false
# ── Konfiguration: Anzeigename des DIZ → Ordner mit den FHIRScan-Ausgaben ────
from pathlib import Path

BASE = Path("..")   # Projekt-Root (relativ zu diesem Notebook)

DIZ = {
    # "Würzburg Testserver":    BASE / "diz_profiles" / "Würzburg_01",
    "Würzburg":                 BASE / "diz_profiles" / "01_Würzburg",
    # "LMU synth":              BASE / "diz_profiles" / "LMU_synth",
    "LMU":                      BASE / "diz_profiles" / "02_LMU",
    "Synthetische Daten":       BASE / "diz_profiles" / "03_Synthetic",
}

# Schwellen, die im Bericht erklärt werden
AEHNLICH_PP = 0.10          # Felder gelten als „ähnlich befüllt“, wenn sich die DIZ um höchstens 10 Prozentpunkte unterscheiden
TOP_WERTE_JE_FELD = 10      # je DIZ gezeigte häufigste Werte pro Feld (die Excel-Tabelle enthält alle gelieferten)
JAHR_MIN_PLAUSIBEL = 1900

In [ ]:
#| include: false
import os, re, json, html
from datetime import date, datetime
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.offline

STAND = date.today().strftime("%d.%m.%Y")

# Programmversion ab Format 2: Datei 00 enthält „Version: … (output format N)“.
# Ältere Läufe (Format 1) werden genauso gelesen; nur die Jahreszählung fehlt dort.
DEFAULT_INTEGRITY = {"Patient", "Encounter", "Practitioner", "PractitionerRole",
                     "Organization", "Device", "RelatedPerson", "Coverage"}
DEFAULT_DEDUP = {"Medication", "Location"}
OHNE_PATIENTENBEZUG = {"Patient", "Medication", "Location", "Organization", "Practitioner",
                       "PractitionerRole", "Substance", "Device"}
OHNE_ENCOUNTERBEZUG = OHNE_PATIENTENBEZUG | {"Encounter", "Consent", "Specimen"}


def lese_lauf(folder: Path) -> dict:
    """Liest die Eckdaten des Laufs aus 00_run_summary.txt (fehlt bei sehr alten Läufen)."""
    p = folder / "00_run_summary.txt"
    info = {"modus": "unbekannt", "format": 1, "Dedup types": DEFAULT_DEDUP, "Integrity types": DEFAULT_INTEGRITY}
    if not p.exists():
        return info
    t = p.read_text(encoding="utf-8", errors="ignore")
    def feld(name):
        m = re.search(rf"^{re.escape(name)}:\s*(.+)$", t, re.MULTILINE)
        return m.group(1).strip() if m else None
    if feld("Server"):
        info["modus"] = "Server"
    elif feld("Directory"):
        info["modus"] = "Dateien"
    if feld("Started"):
        info["datum"] = datetime.strptime(feld("Started")[:10], "%Y-%m-%d").date()
    info["status"] = feld("Status")
    v = feld("Version")
    if v:
        info["version"] = v.split(" ")[0]
        m = re.search(r"output format (\d+)", v)
        info["format"] = int(m.group(1)) if m else 2
    info["top_n"] = feld("Top-N values")
    for key, default in (("Dedup types", DEFAULT_DEDUP), ("Integrity types", DEFAULT_INTEGRITY)):
        val = feld(key)
        info[key] = default if val is None else ({x.strip() for x in val.split(",")} - {"(none)"})
    return info


def lese_txt(path: Path) -> dict:
    """Liest die eingerückten Textberichte 05 bis 07 in ein verschachteltes Dict."""
    if not path.exists():
        return {}
    root, stack = {}, [(-1, {})]
    stack[0] = (-1, root)
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines()[2:]:
        if not line.strip() or line.strip().startswith("- "):
            continue
        ebene = (len(line) - len(line.lstrip())) // 2
        inhalt = line.strip()
        while stack[-1][0] >= ebene:
            stack.pop()
        eltern = stack[-1][1]
        if inhalt.endswith(":") and ": " not in inhalt:
            neu = {}
            eltern[inhalt[:-1]] = neu
            stack.append((ebene, neu))
        elif ": " in inhalt:
            k, v = inhalt.split(": ", 1)
            eltern[k] = v
    return root


def ist_datenfeld(typen) -> bool:
    """Reine Container-Knoten (nur object/array) sind keine Datenfelder."""
    if pd.isna(typen) or str(typen).strip() == "":
        return False
    return bool({t.strip() for t in str(typen).split(",")} - {"object", "array"})


def lese_felder(folder: Path, name: str):
    dfs = []
    # *_by_<feld>_fields.csv (FHIRScan v2) enthalten die Gruppen, nicht den Ressourcentyp als Ganzes
    for f in sorted(p for p in folder.glob("*_fields.csv") if "_by_" not in p.stem):
        df = pd.read_csv(f)
        df["resource_type"] = f.stem.replace("_fields", "")
        dfs.append(df)
    quelle = "vollständig"
    if not dfs and (folder / "02_summary_details.csv").exists():
        # Nur die Übersicht wurde geliefert: Befüllung und Typen ja, Wertverteilungen nein
        df = pd.read_csv(folder / "02_summary_details.csv").rename(
            columns={"has_type_inconsistency": "type_inconsistency"})
        dfs, quelle = [df], "nur Übersicht (ohne Werte)"
    if not dfs:
        return pd.DataFrame(), "fehlt"
    df = pd.concat(dfs, ignore_index=True)
    for c in ("top_values", "value_count", "unique_count", "unique_count_approximate", "detected_fhir_type"):
        if c not in df.columns:
            df[c] = np.nan
    df["diz"] = name
    # Von den DIZ entfernte Werte (REMOVED, [[entfernt]]) zählen als „keine Werte“
    df.loc[df["top_values"].astype(str).str.strip().isin(["REMOVED", "[[entfernt]]"]), "top_values"] = np.nan
    # Befüllungsgrad aus den Zählern statt aus der auf 2 Stellen gerundeten Spalte presence_rate
    df["presence"] = df["resources_with_field"] / df["total_resources"]
    df["type_inconsistency"] = df["type_inconsistency"].astype(str).str.lower() == "true"
    df["datenfeld"] = df["detected_python_types"].apply(ist_datenfeld)
    df = df[df["field_path"] != df["resource_type"]]     # die Ressource selbst ist kein Feld
    return df, quelle


def werte_nenner(r):
    """Anzahl der einzelnen Werte eines Feldes. Läufe mit Ausgabeformat 1 und 2 haben bei Listen jede Liste zusätzlich als
    einen Wert gezählt (value_count = Listen + Einträge). Bei einfachen Listen (z.B. meta.profile[]) zieht diese Funktion
    die Listen ab. Bei Listen in Listen ist die Zahl der Listen nicht bekannt; dann zählt die Summe der erfassten Top-Werte.
    Ab Format 3 zählt value_count nur noch Einträge, dort ist nichts zu korrigieren."""
    vc = r.value_count
    if LAUF[r.diz].get("format", 1) >= 3:
        return vc
    if "array" not in str(r.detected_python_types) or not r.field_path.endswith("[]"):
        return vc
    if r.field_path.count("[]") == 1:
        return vc - r.resources_with_field          # genau eine Liste je Ressource mit dem Feld
    return sum(n for _, n in top_werte(r.top_values)) or vc


def top_werte(s) -> list:
    if pd.isna(s) or s == "":
        return []
    try:
        return [(str(k), int(v)) for k, v in json.loads(s)]
    except Exception:
        return []


# ── Laden ────────────────────────────────────────────────────────────────────
LAUF, QUELLE, REFS, ZEIT, STRUKTUR = {}, {}, {}, {}, {}
summary, card, quality, fields = [], [], [], []
for name, folder in DIZ.items():
    if not folder.exists():
        raise FileNotFoundError(f"Ordner für {name} fehlt: {folder}")
    LAUF[name] = lese_lauf(folder)
    for liste, datei in ((summary, "01_summary.csv"), (card, "03_cardinality.csv"), (quality, "04_data_quality.csv")):
        if (folder / datei).exists():
            d = pd.read_csv(folder / datei)
            d["diz"] = name
            liste.append(d)
    f, QUELLE[name] = lese_felder(folder, name)
    fields.append(f)
    ref_datei = next(iter(sorted(folder.glob("05_reference_integrity*.txt"))), folder / "fehlt")
    REFS[name] = lese_txt(ref_datei).get("references_by_path", {})
    ZEIT[name] = lese_txt(folder / "06_temporal_analysis.txt")
    STRUKTUR[name] = lese_txt(folder / "07_structural_analysis.txt").get("depth_by_type", {})

DIZ_NAMEN = list(DIZ)
S = pd.concat(summary, ignore_index=True)
C = pd.concat(card, ignore_index=True) if card else pd.DataFrame()
Q = pd.concat(quality, ignore_index=True) if quality else pd.DataFrame()
F = pd.concat(fields, ignore_index=True)
F = F[F.datenfeld]          # nur Felder mit eigenem Wert; Gruppen-Elemente wie address[] bleiben außen vor

PATIENTEN = {d: int(S[(S.diz == d) & (S.resource_type == "Patient")]["total_resources"].sum()) for d in DIZ_NAMEN}
GELIEFERT = {rt: [d for d in DIZ_NAMEN if ((S.diz == d) & (S.resource_type == rt) & (S.total_resources > 0)).any()]
             for rt in sorted(S.resource_type.unique())}
ALLE_RT = list(GELIEFERT)
MIT_WERTEN = [d for d in DIZ_NAMEN if F[F.diz == d]["top_values"].notna().any()]
OHNE_WERTE = [d for d in DIZ_NAMEN if d not in MIT_WERTEN]

In [ ]:
#| include: false
# ── Darstellungshilfen ──────────────────────────────────────────────────────
FARBEN_DIZ = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#56B4E9", "#6B6B6B"]   # gut unterscheidbar, auch bei Farbsehschwäche
FARBE = {d: FARBEN_DIZ[i % len(FARBEN_DIZ)] for i, d in enumerate(DIZ_NAMEN)}
GUT = "RdYlGn"      # Kennzahlen, bei denen mehr besser ist (Befüllung, Verknüpfung): rot = 0 %, grün = 100 %
NEUTRAL = "YlGnBu"  # reine Verteilungen (Anteil eines Codes, Ressourcen je Patient)
GRAU_NV = "#E4E8EB"
SCHRIFT = 'Aptos, "Segoe UI", Arial, sans-serif'


def zahl(n) -> str:
    return "n.v." if n is None or pd.isna(n) else f"{int(round(n)):,}".replace(",", ".")


def dez(x, stellen=1) -> str:
    return "n.v." if x is None or pd.isna(x) else f"{x:,.{stellen}f}".replace(",", " ").replace(".", ",").replace(" ", ".")


def proz(p) -> str:
    if p is None or pd.isna(p):
        return "n.v."
    if 0 < p < 0.005:
        return "<1 %"
    if 0.995 <= p < 1:
        return ">99 %"
    return f"{p * 100:.0f} %"


def roh_html(s: str) -> None:
    print("\n`````{=html}\n" + s + "\n`````\n")


def zeige(fig) -> None:
    fig.update_layout(font=dict(family=SCHRIFT, size=12, color="#262626"), separators=",.",
                      paper_bgcolor="white", plot_bgcolor="white", hoverlabel=dict(font_family=SCHRIFT))
    roh_html(fig.to_html(full_html=False, include_plotlyjs=False,
                         config={"displaylogo": False, "responsive": True,
                                 "modeBarButtonsToRemove": ["lasso2d", "select2d", "autoScale2d"]}))


def quelle(text: str) -> None:
    print(f'\n[Quelle: {text}]{{.quelle}}\n')


def erklaerung(gemessen: str, lesart: str, belastbarkeit: str, titel="Wie entsteht diese Darstellung?") -> None:
    print(f'\n::: {{.callout-note collapse="true" icon="false" title="{titel}"}}\n'
          f"**Was gemessen wurde:** {gemessen}\n\n**Wie zu lesen:** {lesart}\n\n"
          f"**Wie belastbar:** {belastbarkeit}\n:::\n")


def cave(text: str, titel="Einschränkung") -> None:
    print(f'\n::: {{.callout-warning icon="false" title="{titel}"}}\n{text}\n:::\n')


def tabelle(df: pd.DataFrame, auffaellig=None) -> None:
    """HTML-Tabelle im BHC-Stil. auffaellig: DataFrame gleicher Form mit True für hervorzuhebende Zellen."""
    kopf = "".join(f"<th>{html.escape(str(c))}</th>" for c in df.columns)
    zeilen = []
    for i, (_, r) in enumerate(df.iterrows()):
        zellen = []
        for j, v in enumerate(r):
            cls = ' class="auffaellig"' if auffaellig is not None and bool(auffaellig.iat[i, j]) else ""
            zellen.append(f"<td{cls}>{html.escape('' if pd.isna(v) else str(v))}</td>")
        zeilen.append("<tr>" + "".join(zellen) + "</tr>")
    roh_html(f'<table class="bhc"><thead><tr>{kopf}</tr></thead><tbody>{"".join(zeilen)}</tbody></table>')


def tabs(eintraege: dict) -> None:
    """Reiter je Ressourcentyp o.ä.: {Reitername: Funktion, die den Inhalt ausgibt}."""
    if not eintraege:
        return
    print("\n::: {.panel-tabset}\n")
    for name, inhalt in eintraege.items():
        print(f"\n#### {name} {{.unnumbered .unlisted}}\n")
        inhalt()
    print("\n:::\n")


def kurz(pfad: str, rt: str) -> str:
    """Feldpfad ohne Ressourcentyp und mit gekürzten Extension-URLs."""
    p = pfad[len(rt) + 1:] if pfad.startswith(rt + ".") else pfad
    return re.sub(r"extension\[[^\]]*?/([^/\]]+)\]", r"extension[\1]", p)


def heatmap(z: pd.DataFrame, text, hover, zeilenhoehe=24, skala=True, log=False, legende="befüllt",
            farben=GUT, spalten=None):
    """Werte 0..1 (oder log), NaN wird als graues Feld „n.v.“ dargestellt."""
    werte = z.values.astype(float)
    nv = np.where(np.isnan(werte), 1.0, np.nan)
    nv_text = np.where(np.isnan(werte), np.where(text == "", "n.v.", text), "")
    x = spalten or list(z.columns)
    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=werte, x=x, y=list(z.index), text=text, texttemplate="%{text}",
                             customdata=hover, hovertemplate="%{customdata}<extra></extra>",
                             colorscale=farben, zmin=None if log else 0, zmax=None if log else 1,
                             xgap=2, ygap=2, showscale=skala and not log, textfont=dict(size=11),
                             colorbar=dict(tickformat=".0%", thickness=12, len=min(1, 300 / (60 + zeilenhoehe * len(z))),
                                           y=1, yanchor="top", title=dict(text=legende, side="top"))))
    fig.add_trace(go.Heatmap(z=nv, x=x, y=list(z.index), text=nv_text, texttemplate="%{text}",
                             customdata=hover, hovertemplate="%{customdata}<extra></extra>",
                             colorscale=[[0, GRAU_NV], [1, GRAU_NV]], showscale=False, xgap=2, ygap=2,
                             textfont=dict(size=11, color="#4A5568")))
    fig.update_layout(height=max(180, 70 + zeilenhoehe * len(z)), margin=dict(l=10, r=10, t=40, b=10),
                      xaxis=dict(side="top", tickfont=dict(size=12)),
                      yaxis=dict(autorange="reversed", automargin=True, tickfont=dict(size=11),
                                 tickmode="array", tickvals=list(z.index)))
    return fig


def befuellungs_heatmap(P: pd.DataFrame, n_mit: pd.DataFrame, n_ges: pd.DataFrame, labels, volle_namen, spalten=None):
    text = np.vectorize(lambda p: "" if pd.isna(p) else proz(p))(P.values)
    hover = np.empty(P.shape, dtype=object)
    for i in range(P.shape[0]):
        for j, d in enumerate(P.columns):
            p = P.iat[i, j]
            hover[i, j] = (f"<b>{d}</b><br>{volle_namen[i]}<br>" +
                           ("in keiner Ressource vorhanden" if pd.isna(p) else
                            f"befüllt: {dez(p * 100, 2)} %<br>{zahl(n_mit.iat[i, j])} von {zahl(n_ges.iat[i, j])} Ressourcen"))
    Z = P.copy()
    Z.index = labels
    return heatmap(Z, text, hover, spalten=spalten)

In [ ]:
#| include: false
# ── Kennzahlen: Heterogenitätsklassen der Felder ────────────────────────────
K1, K2, K3, K4 = ("Bei allen vorhanden, ähnlich befüllt", "Bei allen vorhanden, unterschiedlich befüllt",
                  "Nur bei einem Teil der DIZ vorhanden", "Nur bei einem DIZ vorhanden")
KLASSEN_FARBEN = {K1: "#1A9850", K2: "#A6D96A", K3: "#FDAE61", K4: "#D73027"}

def pivot(df, wert):
    return df.pivot_table(index=["resource_type", "field_path"], columns="diz", values=wert, aggfunc="first").reindex(columns=DIZ_NAMEN)

PIV, N_MIT, N_GES = pivot(F, "presence"), pivot(F, "resources_with_field"), pivot(F, "total_resources")

klassen = []
for (rt, fp), row in PIV.iterrows():
    geliefert = GELIEFERT.get(rt, [])
    werte = row[geliefert].dropna()
    spanne = werte.max() - werte.min() if len(werte) >= 2 else np.nan
    if len(geliefert) < 2:
        k = "Ressourcentyp nur bei einem DIZ"
    elif len(werte) == len(geliefert):
        k = K1 if spanne <= AEHNLICH_PP else K2
    elif len(werte) == 1:
        k = K4
    else:
        k = K3
    klassen.append({"resource_type": rt, "field_path": fp, "klasse": k, "spanne": spanne,
                    "bei_diz": len(werte), "geliefert": len(geliefert)})
KL = pd.DataFrame(klassen)
VERGL = KL[KL.klasse != "Ressourcentyp nur bei einem DIZ"]
NUR_EIN_DIZ_RT = {rt: g[0] for rt, g in GELIEFERT.items() if len(g) == 1}

# Platzhalter für unbekannte Werte (data-absent-reason)
DAR = F[F.field_path.str.contains("data-absent-reason", regex=False)].copy()
if not DAR.empty:
    DAR["element"] = (DAR.field_path.str.replace(r"\.extension\[[^\]]*data-absent-reason\].*$", "", regex=True)
                      .str.replace(r"\._([A-Za-z]+)$", r".\1", regex=True))
    DAR = DAR[~DAR.field_path.str.endswith("].url")]

# Referenzen: nur prüfbar, wenn die Ziel-IDs im Lauf gespeichert wurden
ref_zeilen = []
for d in DIZ_NAMEN:
    lauf = LAUF[d]
    alle_gespeichert = lauf["modus"] == "Dateien"
    gespeichert = lauf["Dedup types"] | lauf["Integrity types"]
    geliefert_rt = set(S[(S.diz == d) & (S.total_resources > 0)].resource_type)
    for key, e in REFS[d].items():
        if " -> " not in key:
            continue
        quelle_pfad, ziel = key.rsplit(" -> ", 1)
        ges, ok, orphan = int(e.get("total", 0)), int(e.get("valid", 0)), int(e.get("orphan", 0))
        ohne_typ = int(e.get("unknown_type", 0))
        if ziel == "unknown":
            status = "ohne Typangabe"
        elif not alle_gespeichert and ziel not in gespeichert:
            status = "nicht geprüft"
        elif ziel not in geliefert_rt:
            status = "Ziel nicht im Datenbestand"
        else:
            status = "geprüft"
        ref_zeilen.append({"diz": d, "pfad": quelle_pfad, "ziel": ziel, "gesamt": ges, "gueltig": ok,
                           "nicht_gefunden": orphan if status == "geprüft" else np.nan, "status": status,
                           "anteil_gueltig": ok / (ok + orphan) if status == "geprüft" and ok + orphan else np.nan})
R = pd.DataFrame(ref_zeilen)

# Zeitraum
def jahr(s):
    m = re.match(r"^(\d{4})", str(s))
    return int(m.group(1)) if m else None

zeit_zeilen, jahr_zeilen = [], []
for d in DIZ_NAMEN:
    lauf_jahr = (LAUF[d].get("datum") or date.today()).year
    for rt, e in ZEIT[d].get("date_ranges_by_type", {}).items():
        j0, j1 = jahr(e.get("earliest")), jahr(e.get("latest"))
        zeit_zeilen.append({"diz": d, "resource_type": rt, "anzahl": int(e.get("count", 0)),
                            "frueh": e.get("earliest", "")[:10], "spaet": e.get("latest", "")[:10],
                            "j0": j0, "j1": j1,
                            "unplausibel": (j0 is not None and j0 < JAHR_MIN_PLAUSIBEL) or (j1 is not None and j1 > lauf_jahr + 1)})
    for rt, jahre in ZEIT[d].get("year_counts_by_type", {}).items():
        for j, n in jahre.items():
            jahr_zeilen.append({"diz": d, "resource_type": rt, "jahr": int(j), "anzahl": int(n)})
T = pd.DataFrame(zeit_zeilen)
J = pd.DataFrame(jahr_zeilen)
MIT_JAHREN = sorted(J.diz.unique(), key=DIZ_NAMEN.index) if not J.empty else []

# Profile aus meta.profile[]
PROFIL_BEREICHE = [("medizininformatik-initiative.de/fhir/core", "MII Kerndatensatz"),
                   ("medizininformatik-initiative.de/fhir/ext", "MII Erweiterungsmodul"),
                   ("medizininformatik-initiative.de/", "MII sonstige"), ("gematik.de/fhir/isik", "ISiK"),
                   ("kbv.de/", "KBV"), ("hl7.org/", "HL7")]
prof = []
for _, r in F[F.field_path.str.endswith(".meta.profile[]")].iterrows():
    for url, n in top_werte(r.top_values):
        basis, version = url.rsplit("|", 1) if "|" in url else (url, "ohne Version")
        prof.append({"diz": r.diz, "resource_type": r.resource_type, "profil": basis.rstrip("/").split("/")[-1],
                     "url": basis, "version": version, "anzahl": n, "anteil": n / r.total_resources,
                     "bereich": next((b for m, b in PROFIL_BEREICHE if m in basis), "Sonstige")})
PR = pd.DataFrame(prof)

In [ ]:
#| output: asis
roh_html(f"<script>{plotly.offline.get_plotlyjs()}</script>")
roh_html('<img class="bhc-logo" src="assets/bhc-logo.png" alt="Bavarian Health Cloud GmbH">')
daten = [LAUF[d].get("datum") for d in DIZ_NAMEN if LAUF[d].get("datum")]
print(f"Stand {STAND} · Version 1.0 · {len(DIZ_NAMEN)} Datenintegrationszentren (DIZ)\n")

# Überblick

In [ ]:
#| output: asis
n_rt_alle = sum(1 for g in GELIEFERT.values() if len(g) == len(DIZ_NAMEN))
anteile = VERGL.klasse.value_counts(normalize=True)
fakten = [
    f"Verglichen werden {len(DIZ_NAMEN)} DIZ mit zusammen {zahl(sum(PATIENTEN.values()))} Patienten "
    f"und {zahl(S.total_resources.sum())} Ressourcen.",
    f"Insgesamt kommen {len(ALLE_RT)} Ressourcentypen vor, davon {n_rt_alle} bei allen DIZ.",
]
if not VERGL.empty:
    fakten.append(f"Von {zahl(len(VERGL))} vergleichbaren Feldern sind {proz(anteile.get(K1, 0))} bei allen DIZ vorhanden "
                  f"und ähnlich befüllt, {proz(anteile.get(K2, 0))} bei allen vorhanden, aber unterschiedlich befüllt, "
                  f"und {proz(anteile.get(K3, 0) + anteile.get(K4, 0))} nur bei einem Teil der DIZ vorhanden.")
    top = VERGL.dropna(subset=["spanne"]).sort_values("spanne", ascending=False).head(1)
    if not top.empty:
        t = top.iloc[0]
        werte = ", ".join(f"{d} {proz(PIV.loc[(t.resource_type, t.field_path), d])}" for d in GELIEFERT[t.resource_type])
        fakten.append(f"Der größte Unterschied im Befüllungsgrad besteht bei {t.resource_type}.{kurz(t.field_path, t.resource_type)} ({werte}).")
if not T.empty and T.unplausibel.any():
    betroffen = T[T.unplausibel].groupby("diz").resource_type.apply(lambda s: ", ".join(sorted(s)))
    fakten.append("Unplausible Datumsangaben (vor 1900 oder mehr als ein Jahr nach dem Lauf) gibt es bei " +
                  "; ".join(f"{d}: {v}" for d, v in betroffen.items()) + ".")
if not PR.empty and PR.diz.nunique() >= 2:
    dom = PR.sort_values("anzahl").groupby(["resource_type", "profil", "diz"]).tail(1)
    abw = dom.groupby(["resource_type", "profil"]).version.nunique()
    n_abw = int((abw > 1).sum())
    fakten.append(f"Bei {n_abw} Profilen verwenden die DIZ unterschiedliche Versionen als häufigste Version.")
if OHNE_WERTE:
    fakten.append(f"Für {', '.join(OHNE_WERTE)} liegen nur Übersichtsdaten vor. Wertverteilungen und Profilversionen fehlen dort.")
print("\n".join(f"- {s}" for s in fakten) + "\n")

print('\n::: {.callout-note icon="false" title="Begriffe"}\n'
      "**Ressource:** ein einzelner Datensatz im Format Fast Healthcare Interoperability Resources (FHIR), "
      "z.B. eine Diagnose, ein Laborwert oder ein Krankenhausaufenthalt.\n\n"
      "**Ressourcentyp:** die Art der Ressource, z.B. *Condition* (Diagnose), *Observation* (Befund, Laborwert), "
      "*Encounter* (Fall bzw. Kontakt), *MedicationAdministration* (Medikamentengabe).\n\n"
      "**Feld:** eine einzelne Angabe innerhalb einer Ressource, z.B. das Datum oder der Code einer Diagnose. "
      "Felder werden als Pfad geschrieben, z.B. `code.coding[].system`; `[]` steht für eine Liste mehrerer Einträge.\n\n"
      "**Befüllungsgrad:** Anteil der Ressourcen eines Typs, in denen ein Feld vorkommt.\n\n"
      "**FHIRScan** speichert Zählungen, Anteile und die häufigsten Werte je Feld, aber keine vollständigen Datensätze.\n:::\n")

## Datengrundlage je DIZ

In [ ]:
#| output: asis
zeilen = []
for d in DIZ_NAMEN:
    l = LAUF[d]
    zeilen.append({"DIZ": d, "Erhebung": {"Server": "FHIR-Server", "Dateien": "Dateiexport"}.get(l["modus"], "unbekannt"),
                   "Lauf vom": l["datum"].strftime("%d.%m.%Y") if l.get("datum") else "unbekannt",
                   "Patienten": zahl(PATIENTEN[d]), "Ressourcen": zahl(S[S.diz == d].total_resources.sum()),
                   "Ressourcentypen": int((S[S.diz == d].total_resources > 0).sum()),
                   "Felddaten": QUELLE[d], "Programmversion": l.get("version", "vor 2026.11")})
tabelle(pd.DataFrame(zeilen))
quelle("00_run_summary.txt, 01_summary.csv, *_fields.csv")
erklaerung(
    "Eckdaten jedes FHIRScan-Laufs. „FHIR-Server“ heißt, dass FHIRScan die Daten Patient für Patient direkt vom Server "
    "des DIZ abgerufen hat. „Dateiexport“ heißt, dass ein Export als Dateien ausgewertet wurde.",
    "Die Tabelle zeigt, worauf alle folgenden Vergleiche beruhen. Unterschiede in Erhebungsart, Zeitpunkt und "
    "Programmversion können selbst Unterschiede in den Ergebnissen erklären.",
    "Beim Abruf vom Server werden nur Ressourcen erfasst, die einem Patienten zugeordnet sind. Eigenständige Ressourcen wie "
    "Organisationen oder Ärzte fehlen dann. Läufe älterer Programmversionen sind voll vergleichbar; nur die Verteilung über "
    "die Jahre fehlt dort (Kapitel 7).", titel="Wie ist diese Tabelle zu verstehen?")

# Welche Daten liefern die DIZ?

## Ressourcen je Patient

In [ ]:
#| output: asis
Z = S.pivot_table(index="resource_type", columns="diz", values="total_resources", aggfunc="sum").reindex(columns=DIZ_NAMEN)
Z = Z.where(Z > 0)
JP = Z.div(pd.Series(PATIENTEN))
JP = JP.loc[JP.max(axis=1).sort_values(ascending=False).index]
text = np.vectorize(lambda v: "" if pd.isna(v) else ("<0,01" if v < 0.005 else dez(v, 2) if v < 10 else dez(v, 0)))(JP.values)
hover = np.array([[f"<b>{d}</b><br>{rt}<br>" + ("nicht geliefert" if pd.isna(JP.at[rt, d]) else
                   f"{dez(JP.at[rt, d], 2)} je Patient<br>{zahl(Z.at[rt, d])} Ressourcen")
                   for d in JP.columns] for rt in JP.index], dtype=object)
zeige(heatmap(np.log10(JP.clip(lower=1e-4)), text, hover, log=True, skala=False, farben=NEUTRAL))
quelle("01_summary.csv, Spalte total_resources geteilt durch die Zahl der Patient-Ressourcen")
erklaerung(
    "Anzahl der Ressourcen eines Typs geteilt durch die Zahl der Patienten des DIZ. „n.v.“ heißt, dass dieser Ressourcentyp "
    "beim DIZ gar nicht vorkommt.",
    "Je dunkler das Feld, desto mehr Ressourcen dieses Typs gibt es pro Patient (von Gelb über Grün nach Blau). Die Farbstufen sind logarithmisch, "
    "damit seltene Typen (unter 0,1 je Patient) und sehr häufige Typen (über 100 je Patient) gleichzeitig unterscheidbar sind. "
    "Die Zahl im Feld ist der genaue Wert.",
    "Die Werte sind exakte Zählungen. Durch die Teilung durch die Patientenzahl werden große und kleine DIZ vergleichbar. "
    "Unterschiede können echte Unterschiede in der Versorgung sein, aber auch daran liegen, welche Quellsysteme ein DIZ "
    "bereits angebunden hat.")
if NUR_EIN_DIZ_RT:
    print("\nNur bei einem DIZ vorhanden: " + ", ".join(f"{rt} ({d})" for rt, d in NUR_EIN_DIZ_RT.items()) + ".\n")

# Wie vollständig sind die Felder befüllt?

## Heterogenität der Felder

In [ ]:
#| output: asis
if VERGL.empty:
    print("Für einen Vergleich werden mindestens zwei DIZ mit gemeinsamen Ressourcentypen benötigt.\n")
else:
    klassen_da = [k for k in (K1, K2, K3, K4) if (VERGL.klasse == k).any()]
    zaehl = VERGL.groupby(["resource_type", "klasse"]).size().unstack(fill_value=0).reindex(columns=klassen_da, fill_value=0)
    zaehl = zaehl.loc[zaehl.sum(axis=1).sort_values().index]
    zaehl.loc["Alle Ressourcentypen"] = zaehl.sum()
    anteil = zaehl.div(zaehl.sum(axis=1), axis=0)
    fig = go.Figure()
    for k in klassen_da:
        fig.add_trace(go.Bar(y=anteil.index, x=anteil[k], name=k, orientation="h", marker_color=KLASSEN_FARBEN[k],
                             text=[str(n) if n else "" for n in zaehl[k]], textposition="inside",
                             customdata=zaehl[k], hovertemplate="%{y}<br>" + k + ": %{customdata} Felder (%{x:.0%})<extra></extra>"))
    fig.update_layout(barmode="stack", height=90 + 28 * len(anteil), margin=dict(l=10, r=10, t=10, b=10),
                      xaxis=dict(tickformat=".0%", range=[0, 1]), yaxis=dict(automargin=True),
                      legend=dict(orientation="h", traceorder="normal", y=-0.12 if len(anteil) > 6 else -0.3))
    zeige(fig)
    quelle("*_fields.csv bzw. 02_summary_details.csv, Spalten resources_with_field und total_resources")
    erklaerung(
        "Für jedes Feld wird geprüft, bei wie vielen der DIZ es vorkommt, die den Ressourcentyp überhaupt liefern. "
        f"Kommt es bei allen vor, wird der Befüllungsgrad verglichen: Bei höchstens {AEHNLICH_PP * 100:.0f} Prozentpunkten "
        "Unterschied gilt es als „ähnlich befüllt“. Die Zahl im Balken ist die Anzahl der Felder. Gezählt werden nur Felder mit "
        "eigenem Wert, Gruppen-Elemente wie `address[]` nicht.",
        "Ein hoher grüner Anteil bedeutet: Die DIZ liefern für diesen Ressourcentyp weitgehend dieselben Angaben in ähnlichem "
        "Umfang. Orange und rote Anteile zeigen Felder, die nur manche DIZ liefern. Ressourcentypen, die nur ein DIZ liefert, "
        "sind hier nicht enthalten.",
        "Ein Feld gilt als vorhanden, sobald es in mindestens einer Ressource vorkommt. Fehlt es, kann das am Quellsystem, an "
        "einer anderen Profilversion oder an einer anderen Erweiterung (Extension) liegen. Dieselbe Information unter einem "
        "anderen Pfad zählt als anderes Feld. Die Schwelle von 10 Prozentpunkten ist eine Festlegung für diesen Bericht.")

## Completeness Map

Jede Zeile ist ein Feld, jede Spalte ein DIZ. Gezeigt werden alle Felder, die bei mindestens einem DIZ vorkommen.

In [ ]:
#| output: asis
def map_fuer(rt):
    def inhalt():
        geliefert = GELIEFERT[rt]
        P = PIV.loc[rt][geliefert].dropna(how="all").sort_index()
        if P.empty:
            print("Keine Felddaten vorhanden.\n")
            return
        idx = [(rt, fp) for fp in P.index]
        n = P.notna().sum()
        print(f"\n{len(P)} verschiedene Felder insgesamt, davon " +
              ", ".join(f"{zahl(n[d])} bei {d}" for d in geliefert) + ".\n")
        labels = [kurz(fp, rt) for fp in P.index]
        zeige(befuellungs_heatmap(P, N_MIT.loc[idx, geliefert], N_GES.loc[idx, geliefert], labels, list(P.index),
                                  spalten=[f"{d}<br>{zahl(n[d])} Felder" for d in geliefert]))
        fehlend = [d for d in DIZ_NAMEN if d not in geliefert]
        if fehlend:
            print(f"\n{rt} kommt bei {', '.join(fehlend)} nicht vor.\n")
    return inhalt

tabs({rt: map_fuer(rt) for rt in ALLE_RT if rt in PIV.index.get_level_values(0)})
quelle("*_fields.csv bzw. 02_summary_details.csv, Spalten resources_with_field und total_resources")
erklaerung(
    "Für jedes Feld der Anteil der Ressourcen, in denen es vorkommt (Befüllungsgrad). Grundlage sind die exakten Zähler, "
    "nicht gerundete Werte.",
    "Grün heißt: Das Feld ist (fast) immer befüllt, Gelb: etwa in der Hälfte der Ressourcen, Rot: selten. „<1 %“ heißt, dass "
    "das Feld zwar vorkommt, aber in weniger als einer von hundert Ressourcen. Grau mit „n.v.“ heißt: Das Feld kommt beim DIZ "
    "in keiner einzigen Ressource vor. Über jeder Spalte steht, wie viele Felder beim DIZ vorkommen. Felder sind nach Pfad "
    "sortiert, sodass zusammengehörige Angaben untereinander stehen.",
    "Die Werte sind exakte Zählungen über alle Ressourcen. Gezeigt werden alle Felder mit eigenem Wert. Elemente, die nur "
    "weitere Felder bündeln (z.B. `address[]` für Straße, Ort und Postleitzahl), sind nicht aufgeführt, ihre Felder schon. "
    "Die Feldzahl liegt deshalb unter der Spalte total_fields in 01_summary.csv, die solche Elemente und die Ressource selbst "
    "mitzählt. FHIRScan kann nur Felder sehen, die mindestens einmal vorkommen. "
    "Ein Feld, das bei allen DIZ fehlt, erscheint deshalb nicht. Ein befülltes Feld sagt nichts darüber, ob der Inhalt "
    "inhaltlich richtig ist.")

## Größte Unterschiede

In [ ]:
#| output: asis
U = VERGL[VERGL.klasse == K2].dropna(subset=["spanne"]).sort_values("spanne", ascending=False).head(30)
if U.empty:
    print("Keine Felder mit deutlich unterschiedlichem Befüllungsgrad.\n")
else:
    labels = [f"{r.resource_type}: {kurz(r.field_path, r.resource_type)}" for r in U.itertuples()]
    fig = go.Figure()
    lx, ly = [], []
    for lab, r in zip(labels, U.itertuples()):
        w = PIV.loc[(r.resource_type, r.field_path)].dropna()
        lx += [w.min(), w.max(), None]
        ly += [lab, lab, None]
    fig.add_trace(go.Scatter(x=lx, y=ly, mode="lines", line=dict(color="#B8C2CC", width=3), hoverinfo="skip", showlegend=False))
    for d in DIZ_NAMEN:
        xs = [PIV.loc[(r.resource_type, r.field_path), d] for r in U.itertuples()]
        fig.add_trace(go.Scatter(x=xs, y=labels, mode="markers", name=d, marker=dict(size=11, color=FARBE[d]),
                                 hovertemplate="%{y}<br>" + d + ": %{x:.1%}<extra></extra>"))
    fig.update_layout(height=100 + 24 * len(U), margin=dict(l=10, r=10, t=10, b=10),
                      xaxis=dict(tickformat=".0%", range=[-0.02, 1.02], title="Befüllungsgrad", gridcolor="#E6EEF7"),
                      yaxis=dict(autorange="reversed", automargin=True, tickfont=dict(size=11)),
                      legend=dict(orientation="h", y=1.02, yanchor="bottom"))
    zeige(fig)
    quelle("*_fields.csv bzw. 02_summary_details.csv")
    erklaerung(
        "Die 30 Felder, die bei allen liefernden DIZ vorkommen und sich im Befüllungsgrad am stärksten unterscheiden. "
        "Gemessen wird die Spanne zwischen dem niedrigsten und dem höchsten Wert.",
        "Jeder Punkt ist ein DIZ. Je länger die Linie, desto größer der Unterschied. Felder, die bei einem DIZ ganz fehlen, "
        "stehen in der Completeness Map.",
        "Exakte Zählungen. Ein großer Unterschied ist ein Hinweis, kein Fehlerbefund: Häufig steckt eine unterschiedliche "
        "Dokumentationspraxis dahinter, z.B. ob ein Datum als Zeitpunkt oder als Zeitraum erfasst wird.")

## Platzhalter für unbekannte Werte

In [ ]:
#| output: asis
if DAR.empty:
    print("In den Felddaten kommen keine Platzhalter für unbekannte Werte vor.\n")
else:
    D2 = DAR.groupby(["resource_type", "element", "diz"]).presence.max().unstack().reindex(columns=DIZ_NAMEN)
    D2 = D2.reset_index()
    D2.insert(1, "Feld", [kurz(e, rt) for rt, e in zip(D2.resource_type, D2.element)])
    out = D2.drop(columns="element").rename(columns={"resource_type": "Ressourcentyp"})
    for d in DIZ_NAMEN:
        out[d] = out[d].map(lambda p: "kein Platzhalter" if pd.isna(p) else proz(p))
    tabelle(out)
    quelle("*_fields.csv, Felder mit der Erweiterung data-absent-reason")
    erklaerung(
        "FHIR erlaubt, statt eines Wertes einen Platzhalter mit Begründung zu speichern („data-absent-reason“, z.B. „unknown“). "
        "Gezeigt wird der Anteil der Ressourcen, die für das Feld einen solchen Platzhalter enthalten.",
        "Hohe Anteile heißen: Das Feld ist formal vorhanden, enthält aber häufig keinen echten Wert.",
        "Exakte Zählungen. In der Completeness Map zählt ein Platzhalter beim Element selbst nicht als Wert. Er erscheint dort "
        "als eigenes Feld mit „extension[data-absent-reason]“ im Pfad.")

# Wie gut sind die Daten verknüpft?

## Bezug zu Patient und Encounter

In [ ]:
#| output: asis
def link_map(spalte, ausschluss):
    if Q.empty:
        print("Keine Daten zur Verknüpfung vorhanden.\n")
        return
    L = Q.pivot_table(index="resource_type", columns="diz", values=spalte, aggfunc="first").reindex(columns=DIZ_NAMEN)
    L = L.drop(index=[r for r in L.index if r in ausschluss])
    for rt in L.index:
        for d in DIZ_NAMEN:
            if d not in GELIEFERT.get(rt, []):
                L.at[rt, d] = np.nan
    text = np.vectorize(lambda p: "" if pd.isna(p) else proz(p))(L.values)
    hover = np.array([[f"<b>{d}</b><br>{rt}<br>" + ("nicht geliefert" if pd.isna(L.at[rt, d]) else f"{dez(L.at[rt, d] * 100, 2)} %")
                       for d in L.columns] for rt in L.index], dtype=object)
    zeige(heatmap(L, text, hover, legende="mit Bezug"))

tabs({"Bezug zum Patienten": lambda: link_map("patient_linkage_rate", OHNE_PATIENTENBEZUG),
      "Bezug zum Encounter (Fall)": lambda: link_map("encounter_linkage_rate", OHNE_ENCOUNTERBEZUG)})
quelle("04_data_quality.csv")
erklaerung(
    "Anteil der Ressourcen, die mindestens einen Verweis auf einen Patienten bzw. auf einen Encounter (Fall oder Kontakt) "
    "enthalten. Gezählt wird jeder Verweis in der Ressource, z.B. in `subject`, `patient` oder `encounter`.",
    "Klinische Daten sollten fast immer einem Patienten zugeordnet sein; Werte deutlich unter 100 % sind auffällig. Beim "
    "Bezug zum Encounter sind niedrigere Werte normal, wenn ein Datensatz nicht an einen Aufenthalt gebunden ist. Ressourcentypen "
    "ohne vorgesehenen Bezug (z.B. Patient, Medication) sind ausgeblendet.",
    "Exakte Zählungen. Geprüft wird nur, ob ein Verweis vorhanden ist, nicht ob das Ziel existiert. Das zeigt der nächste "
    "Abschnitt. Beim Abruf vom Server sind Ressourcen ohne Patientenbezug gar nicht sichtbar, die Rate liegt dort deshalb "
    "methodisch bedingt nahe 100 %.")

## Gültigkeit der Verweise

In [ ]:
#| output: asis
if R.empty:
    print("Keine Daten zur Gültigkeit der Verweise vorhanden.\n")
else:
    RR = R[R.status != "ohne Typangabe"].copy()
    RR["zeile"] = RR.pfad.map(lambda p: kurz(p, "")) + " → " + RR.ziel
    reihen = (RR.assign(g=RR.status == "geprüft").groupby("zeile").g.any().sort_values(ascending=False)
              .index.tolist())
    reihen = sorted(reihen, key=lambda z: (not RR[RR.zeile == z].status.eq("geprüft").any(), z))
    Z = RR.pivot_table(index="zeile", columns="diz", values="anteil_gueltig", aggfunc="first").reindex(index=reihen, columns=DIZ_NAMEN)
    status = RR.pivot_table(index="zeile", columns="diz", values="status", aggfunc="first").reindex(index=reihen, columns=DIZ_NAMEN)
    ges = RR.pivot_table(index="zeile", columns="diz", values="gesamt", aggfunc="first").reindex(index=reihen, columns=DIZ_NAMEN)
    text = np.empty(Z.shape, dtype=object)
    hover = np.empty(Z.shape, dtype=object)
    for i, z in enumerate(reihen):
        for j, d in enumerate(DIZ_NAMEN):
            s, p = status.iat[i, j], Z.iat[i, j]
            text[i, j] = proz(p) if s == "geprüft" else ("" if pd.isna(s) else {"nicht geprüft": "nicht geprüft",
                                                                               "Ziel nicht im Datenbestand": "Ziel fehlt"}[s])
            hover[i, j] = f"<b>{d}</b><br>{z}<br>" + ("kein Verweis dieser Art" if pd.isna(s) else
                                                        f"{zahl(ges.iat[i, j])} Verweise<br>{s}" + (f": {dez(p * 100, 2)} % gefunden" if s == "geprüft" else ""))
    zeige(heatmap(Z, text, hover, legende="gefunden"))
    quelle("05_reference_integrity.txt")
    erklaerung(
        "Für jede Art von Verweis (z.B. von einer Diagnose auf den zugehörigen Encounter) der Anteil der Verweise, deren Ziel im "
        "ausgewerteten Datenbestand tatsächlich gefunden wurde.",
        "100 % heißt: Alle Verweise führen zu einer vorhandenen Ressource. Niedrigere Werte heißen, dass Ziele fehlen, also "
        "Verweise ins Leere gehen. „Ziel fehlt“: Ressourcen dieses Zieltyps kommen im ausgewerteten Bestand gar nicht vor. "
        "„nicht geprüft“: Für diesen Zieltyp hat FHIRScan keine IDs gespeichert, eine Prüfung war nicht möglich.",
        "Beim Abruf vom Server prüft FHIRScan aus Speichergründen nur Verweise auf ausgewählte Zieltypen (u.a. Patient, Encounter, "
        "Medication). Ältere Programmversionen haben die übrigen Verweise fälschlich als ungültig gezählt; dieser Bericht weist sie "
        "für alle Versionen als „nicht geprüft“ aus. Ziele wie Organisationen sind beim Abruf vom Server grundsätzlich nicht "
        "enthalten, weil sie keinem Patienten zugeordnet sind.")

# Wie viele Ressourcen hat ein Patient?

In [ ]:
#| output: asis
LOG_TICKS = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

def verteilung(anker, titel):
    def inhalt():
        K = C[C.anchor_type == anker] if not C.empty else C
        if K.empty:
            print("Keine Daten vorhanden.\n")
            return
        rts = sorted(K.resource_type.unique(), key=lambda r: -K[K.resource_type == r]["median"].astype(float).max())
        fig = go.Figure()
        for d in DIZ_NAMEN:
            k = K[K.diz == d].set_index("resource_type").reindex(rts)
            fig.add_trace(go.Box(x=rts, name=d, marker_color=FARBE[d], q1=k.p25, median=k["median"], q3=k.p75,
                                 lowerfence=k["min"], upperfence=k.p90, boxpoints=False,
                                 hovertemplate=None))
        fig.update_layout(boxmode="group", height=460, margin=dict(l=10, r=10, t=10, b=10),
                          yaxis=dict(type="log", title=f"Ressourcen {titel} (logarithmisch)", gridcolor="#E6EEF7",
                                     tickvals=LOG_TICKS, ticktext=[zahl(t) for t in LOG_TICKS]),
                          legend=dict(orientation="h", y=1.02, yanchor="bottom"))
        zeige(fig)
        A = K.pivot_table(index="resource_type", columns="diz", values="coverage_rate", aggfunc="first").reindex(index=rts, columns=DIZ_NAMEN)
        text = np.vectorize(lambda p: "" if pd.isna(p) else proz(p))(A.values)
        hover = np.array([[f"<b>{d}</b><br>{rt}<br>" + ("nicht geliefert" if pd.isna(A.at[rt, d]) else f"{dez(A.at[rt, d] * 100, 1)} % mit mindestens einer Ressource")
                           for d in A.columns] for rt in A.index], dtype=object)
        print(f"\nAnteil {'der Patienten' if anker == 'per_patient' else 'der Encounter'} mit mindestens einer Ressource des Typs:\n")
        zeige(heatmap(A, text, hover, legende="Anteil"))
    return inhalt

tabs({"Je Patient": verteilung("per_patient", "je Patient"), "Je Encounter (Fall)": verteilung("per_encounter", "je Encounter")})
quelle("03_cardinality.csv")
erklaerung(
    "Für jeden Patienten (bzw. Encounter) wird gezählt, wie viele Ressourcen eines Typs ihm zugeordnet sind. Gezeigt wird die "
    "Verteilung über alle Patienten, die mindestens eine solche Ressource haben. Darunter steht, wie groß dieser Anteil ist.",
    "Der Strich in der Box ist der Median: Die Hälfte der Patienten hat weniger, die Hälfte mehr. Die Box umfasst die mittleren "
    "50 % der Patienten, die Linie reicht vom Minimum bis zu dem Wert, den 90 % der Patienten nicht überschreiten. Die Achse ist "
    "logarithmisch, jeder Gitterschritt ist das Zehnfache.",
    "Exakte Zählungen. Median und Perzentile sind robuster als Mittelwerte, weil wenige Patienten mit sehr vielen Ressourcen "
    "(z.B. Langzeitpatienten) den Mittelwert stark verzerren würden. Die Box beschreibt nur Patienten mit mindestens einer "
    "Ressource; wie viele das sind, zeigt die Tabelle darunter.")

# Welche Codes und Werte werden verwendet?

In [ ]:
#| output: asis
SYSTEME = {"http://loinc.org": "LOINC", "http://snomed.info/sct": "SNOMED CT",
           "http://fhir.de/CodeSystem/bfarm/icd-10-gm": "ICD-10-GM", "http://fhir.de/CodeSystem/dimdi/icd-10-gm": "ICD-10-GM (DIMDI)",
           "http://fhir.de/CodeSystem/bfarm/ops": "OPS", "http://fhir.de/CodeSystem/dimdi/ops": "OPS (DIMDI)",
           "http://fhir.de/CodeSystem/bfarm/atc": "ATC", "http://www.whocc.no/atc": "ATC (WHO)",
           "http://fhir.de/CodeSystem/ifa/pzn": "PZN", "http://unitsofmeasure.org": "UCUM",
           "http://fhir.de/CodeSystem/bfarm/alpha-id": "Alpha-ID"}

def system_name(url):
    return SYSTEME.get(url, url if len(url) <= 70 else "…" + url[-68:])

def werte_heatmap(rt, nur_system):
    sub = F[(F.resource_type == rt) & F.top_values.notna() & F.diz.isin(MIT_WERTEN)]
    if nur_system:
        sub = sub[sub.field_path.str.endswith("coding[].system")]
    else:
        sub = sub[(sub.detected_fhir_type == "code") & ~sub.field_path.str.endswith(".system")
                  & ~sub.field_path.str.contains("extension[", regex=False) & ~sub.field_path.str.endswith(".url")
                  & ~sub.field_path.str.contains(r"identifier|\.reference$|\.id$|\.text$|\.display$|\.resourceType$", regex=True)]
    # Felder ohne lesbare Werte (z.B. vor der Lieferung geschwärzte IDs) auslassen
    sub = sub[sub.top_values.map(lambda tv: any(w for w, _ in top_werte(tv)))]
    if sub.empty:
        return False
    zeilen = {}
    for fp, g in sub.groupby("field_path"):
        auswahl = []
        for r in g.itertuples():
            for w, _ in top_werte(r.top_values)[:TOP_WERTE_JE_FELD]:
                if w not in auswahl:
                    auswahl.append(w)
        for w in auswahl:
            lab = f"{kurz(fp, rt)} · {system_name(w) if nur_system else w}"
            zeilen[lab] = {}
            for r in g.itertuples():
                n = dict(top_werte(r.top_values)).get(w)
                nenner = werte_nenner(r)
                zeilen[lab][r.diz] = (n / nenner if n else 0.0, n or 0, nenner, fp, w)
    labels = list(zeilen)
    spalten = [d for d in MIT_WERTEN if d in GELIEFERT[rt]]
    Z = pd.DataFrame([[zeilen[l].get(d, (np.nan,))[0] for d in spalten] for l in labels], index=labels, columns=spalten)
    text = np.vectorize(lambda p: "" if pd.isna(p) else ("0 %" if p == 0 else proz(p)))(Z.values)
    hover = np.empty(Z.shape, dtype=object)
    for i, l in enumerate(labels):
        for j, d in enumerate(spalten):
            e = zeilen[l].get(d)
            hover[i, j] = (f"<b>{d}</b><br>{l}<br>" + ("Feld kommt nicht vor" if e is None else
                           (f"{zahl(e[1])} von {zahl(e[2])} Werten ({dez(e[0] * 100, 2)} %)" if e[1] else
                            "nicht unter den häufigsten Werten") + f"<br>{html.escape(e[4])}"))
    zeige(heatmap(Z, text, hover, zeilenhoehe=22, legende="Anteil", farben=NEUTRAL))
    return True

def verschiedene(rt):
    sub = F[(F.resource_type == rt) & F.diz.isin(MIT_WERTEN) & ((F.detected_fhir_type == "code") | F.field_path.str.endswith("coding[].code"))
            & ~F.field_path.str.contains("extension[", regex=False)
            & ~F.field_path.str.contains(r"identifier|\.reference$|\.id$|\.text$|\.display$|\.resourceType$", regex=True)]
    if sub.empty:
        return
    V = sub.assign(n=sub.apply(lambda r: ("ca. " if str(r.unique_count_approximate).lower() == "true" else "") + zahl(r.unique_count), axis=1))
    V = V.pivot_table(index="field_path", columns="diz", values="n", aggfunc="first").reindex(columns=[d for d in MIT_WERTEN if d in GELIEFERT[rt]])
    V = V.fillna("Feld fehlt").reset_index()
    V["field_path"] = V.field_path.map(lambda p: kurz(p, rt))
    print("\nAnzahl verschiedener Werte je Feld:\n")
    tabelle(V.rename(columns={"field_path": "Feld"}))

if not MIT_WERTEN:
    print("Für keines der DIZ liegen Wertverteilungen vor.\n")
else:
    if OHNE_WERTE:
        cave(f"Für {', '.join(OHNE_WERTE)} liegen keine Wertverteilungen vor (nur Übersichtsdaten geliefert). "
             "Diese DIZ fehlen in diesem Kapitel und in Kapitel 8.", titel="Unvollständige Datengrundlage")
    print("\n## Codesysteme\n")
    tabs({rt: (lambda rt=rt: werte_heatmap(rt, True) or print("Keine codierten Felder.\n")) for rt in ALLE_RT})
    quelle("*_fields.csv, Spalte top_values der Felder …coding[].system")
    erklaerung(
        "Für jedes codierte Feld der Anteil der Codes aus einem bestimmten Codesystem, z.B. LOINC für Laborwerte oder "
        "ICD-10-GM für Diagnosen. Grundlage sind die häufigsten Werte, die FHIRScan je Feld speichert.",
        "Jede Zeile ist ein Codesystem in einem Feld. Nutzen die DIZ unterschiedliche Codesysteme für dasselbe Feld, sind die "
        "Daten nur mit einer Übersetzung (Mapping) gemeinsam auswertbar. Eine Ressource kann mehrere Codes aus verschiedenen "
        "Systemen tragen, die Anteile beziehen sich deshalb auf alle Codes, nicht auf Ressourcen.",
        "Codesysteme gibt es je Feld meist nur wenige, die häufigsten Werte decken sie in der Regel vollständig ab. Die genauen "
        "Zahlen stehen im Tooltip.")
    print("\n## Häufigste Werte\n")
    tabs({rt: (lambda rt=rt: (werte_heatmap(rt, False) or print("Keine codierten Felder.\n"), verschiedene(rt))) for rt in ALLE_RT})
    quelle("*_fields.csv, Spalten top_values, value_count und unique_count")
    erklaerung(
        f"Für Felder mit Codes (z.B. Status, Geschlecht, Laborcode) die je DIZ {TOP_WERTE_JE_FELD} häufigsten Werte und ihr "
        "Anteil an allen Werten des Feldes. Darunter steht, wie viele verschiedene Werte ein Feld beim DIZ hat.",
        "Werte, die bei allen DIZ häufig sind, deuten auf eine gemeinsame Kodierung hin. Werte, die nur bei einem DIZ vorkommen, "
        "zeigen lokale Codes oder eine andere Auswahl. „0 %“ heißt, dass der Wert beim DIZ nicht unter den häufigsten Werten ist.",
        f"FHIRScan speichert je Feld nur die häufigsten Werte (Top-{LAUF[MIT_WERTEN[0]].get('top_n') or 20}). Seltene Werte "
        "sind daher nicht sichtbar, und „0 %“ heißt nicht zwingend, dass der Wert gar nicht vorkommt. Die Anzahl verschiedener "
        "Werte ist ab 10.000 eine Schätzung (gekennzeichnet mit „ca.“, Abweichung typischerweise unter 1 %).")

# Welchen Zeitraum decken die Daten ab?

In [ ]:
#| output: asis
if T.empty:
    print("Keine Zeitangaben vorhanden.\n")
else:
    W = T.assign(z=T.apply(lambda r: f"{r.j0:04d} bis {r.j1:04d}", axis=1)).pivot_table(index="resource_type", columns="diz", values="z", aggfunc="first").reindex(columns=DIZ_NAMEN)
    M = T.pivot_table(index="resource_type", columns="diz", values="unplausibel", aggfunc="first").reindex(index=W.index, columns=DIZ_NAMEN).fillna(False)
    out = W.fillna("keine Datumsangabe").reset_index().rename(columns={"resource_type": "Ressourcentyp"})
    mark = pd.concat([pd.Series(False, index=M.index, name="Ressourcentyp"), M], axis=1).reset_index(drop=True)
    tabelle(out, auffaellig=mark)
    quelle("06_temporal_analysis.txt, date_ranges_by_type")
    erklaerung(
        "Für jeden Ressourcentyp das früheste und das späteste Datum. Verwendet wird das erste vorhandene von "
        "`effectiveDateTime`, `issued`, `authoredOn`, `recordedDate`, `date`, sonst der Beginn eines Zeitraums (`period.start`).",
        f"Orange markiert sind unplausible Jahre: vor {JAHR_MIN_PLAUSIBEL} oder mehr als ein Jahr nach dem Lauf. Solche Werte sind "
        "meist Platzhalter (z.B. 31.12.9999) oder Erfassungsfehler.",
        "Früheste und späteste Werte sind Extremwerte: Ein einziger fehlerhafter Datensatz reicht für eine Markierung. Wie viele "
        "Datensätze betroffen sind, zeigt nur die Jahresverteilung darunter. Für Patient wird kein Datum ausgewertet (das "
        "Geburtsdatum gehört nicht zu den genannten Feldern).")
    print("\n## Verteilung über die Jahre\n")
    if not MIT_JAHREN:
        print("Die Jahresverteilung erfasst FHIRScan erst ab Programmversion 2026.11. Für die vorliegenden Läufe ist sie nicht verfügbar.\n")
    else:
        ohne = [d for d in DIZ_NAMEN if d not in MIT_JAHREN]
        if ohne:
            print(f"Für {', '.join(ohne)} nicht verfügbar (Lauf mit älterer Programmversion).\n")
        def jahre(rt):
            def inhalt():
                j = J[J.resource_type == rt]
                fig = go.Figure()
                hinweise = []
                for d in MIT_JAHREN:
                    jd = j[j.diz == d]
                    if jd.empty:
                        continue
                    ges = jd.anzahl.sum()
                    bis = (LAUF[d].get("datum") or date.today()).year + 1
                    drin = jd[(jd.jahr >= 1990) & (jd.jahr <= bis)].groupby("jahr").anzahl.sum()
                    if not drin.empty:
                        drin = drin.reindex(range(drin.index.min(), drin.index.max() + 1), fill_value=0)
                    drin = drin.rename_axis("jahr").reset_index()
                    raus = ges - drin.anzahl.sum()
                    if raus:
                        hinweise.append(f"{d}: {zahl(raus)} Ressourcen ({proz(raus / ges)}) vor 1990 oder nach {bis}")
                    fig.add_trace(go.Scatter(x=drin.jahr, y=drin.anzahl / ges, name=d, mode="lines+markers",
                                             line=dict(color=FARBE[d], width=2), marker=dict(size=5),
                                             customdata=drin.anzahl, hovertemplate=d + " %{x}: %{y:.1%} (%{customdata:,} Ressourcen)<extra></extra>"))
                fig.update_layout(height=320, margin=dict(l=10, r=10, t=10, b=10),
                                  yaxis=dict(tickformat=".0%", title="Anteil der Ressourcen", gridcolor="#E6EEF7"),
                                  xaxis=dict(dtick=2, gridcolor="#E6EEF7"), legend=dict(orientation="h", y=1.02, yanchor="bottom"))
                zeige(fig)
                if hinweise:
                    print("\nAußerhalb des Diagramms: " + "; ".join(hinweise) + ".\n")
            return inhalt
        tabs({rt: jahre(rt) for rt in sorted(J.resource_type.unique())})
        erklaerung(
            "Anteil der Ressourcen eines Typs je Kalenderjahr, bezogen auf alle datierten Ressourcen dieses Typs beim DIZ.",
            "Die Kurven zeigen, ab wann ein DIZ Daten liefert und ob Jahre fehlen. Weil Anteile statt Anzahlen gezeigt werden, "
            "sind große und kleine DIZ vergleichbar.",
            "Exakte Zählungen. Jahre vor 1990 und nach dem Lauf sind nicht dargestellt, ihre Zahl steht unter dem Diagramm.")

# Welche MII-Profile werden verwendet?

In [ ]:
#| output: asis
if PR.empty:
    print("Keine Profilangaben vorhanden (das Feld meta.profile fehlt oder es liegen keine Wertverteilungen vor).\n")
else:
    zeilen, marks = [], []
    for (rt, profil), g in PR.groupby(["resource_type", "profil"]):
        z = {"Ressourcentyp": rt, "Profil": profil, "Bereich": g.bereich.iloc[0]}
        dominant = set()
        for d in MIT_WERTEN:
            gd = g[g.diz == d].sort_values("anzahl", ascending=False)
            if d not in GELIEFERT.get(rt, []):
                z[d] = "Ressourcentyp fehlt"
            elif gd.empty:
                z[d] = "nicht verwendet"
            else:
                z[d] = "; ".join(f"{r.version} ({proz(r.anteil)})" for r in gd.itertuples())
                dominant.add(gd.iloc[0].version)
        zeilen.append(z)
        marks.append([False, False, False] + [len(dominant) > 1] * len(MIT_WERTEN))
    out = pd.DataFrame(zeilen)
    tabelle(out, auffaellig=pd.DataFrame(marks))
    quelle("*_fields.csv, Spalte top_values des Feldes meta.profile[]")
    erklaerung(
        "Jede FHIR-Ressource kann angeben, nach welchem Profil sie erstellt wurde, z.B. dem MII-Profil „Diagnose“ in der Version "
        "2025.0.0. Gezeigt werden alle Profile mit Version und dem Anteil der Ressourcen, die sie angeben.",
        "Orange markiert sind Profile, bei denen die DIZ unterschiedliche Versionen am häufigsten verwenden. Unterschiedliche "
        "Versionen können sich in Pflichtfeldern und erlaubten Codes unterscheiden. Mehrere Versionen bei einem DIZ zeigen, dass "
        "Daten aus verschiedenen Ladezeitpunkten nebeneinander liegen.",
        "Eine Ressource kann mehrere Profile angeben, die Anteile ergeben deshalb nicht immer 100 %. Die Profilangabe ist eine "
        "Selbstauskunft der Daten; FHIRScan prüft nicht, ob die Ressource das Profil tatsächlich erfüllt.")

# Methodik und Belastbarkeit

**Was FHIRScan misst.** FHIRScan liest jede Ressource vollständig und zählt für jedes Feld, in wie vielen Ressourcen es vorkommt. Zusätzlich werden die häufigsten Werte je Feld, die Zahl verschiedener Werte, Zahlenbereiche, Verweise zwischen Ressourcen und Datumsangaben erfasst. Gespeichert werden diese Zählungen und die häufigsten Werte je Feld, keine vollständigen Datensätze.

**Exakte Zählungen und Schätzungen.** Befüllungsgrade, Ressourcenzahlen, Verknüpfungen, Verteilungen je Patient und Zeiträume beruhen auf exakten Zählungen über den gesamten Datenbestand, nicht auf Stichproben. Geschätzt wird nur die Zahl verschiedener Werte ab 10.000 (HyperLogLog-Verfahren, typische Abweichung unter 1 %). Die Wertverteilungen sind auf die häufigsten Werte je Feld begrenzt. Bei Feldern, die Kennungen enthalten können (z.B. Identifier, Ressourcen-IDs, Verweise), haben die DIZ die Werte vor der Weitergabe entfernt; diese Felder erscheinen in den Wertverteilungen nicht.

**Was „vorhanden“ bedeutet.** Ein Feld gilt als vorhanden, wenn es in der Ressource mit einem Wert vorkommt. Ob der Wert inhaltlich richtig, aktuell oder fachlich sinnvoll ist, prüft FHIRScan nicht. Ein Feld, das bei keinem DIZ vorkommt, ist für FHIRScan unsichtbar. Für einen Abgleich mit dem Soll (z.B. Pflichtfelder des MII-Kerndatensatzes) wird eine Referenzliste benötigt.

**Vergleichbarkeit der Felder.** Felder werden über ihren Pfad verglichen, z.B. `Condition.code.coding[].system`. Erweiterungen (Extensions) stehen mit ihrer URL im Pfad. Dieselbe Information unter einer anderen Erweiterung oder Profilversion erscheint daher als eigenes Feld. Unterschiede in den Kapiteln 3 und 8 hängen oft zusammen.

**Server und Dateiexport.** Beim Abruf vom FHIR-Server holt FHIRScan die Daten je Patient ab (Operation `$everything`). Ressourcen ohne Patientenbezug (z.B. Organisationen, Ärzte, nicht zugeordnete Medikamente) sind dabei nicht enthalten. Ressourcen, die bei mehreren Patienten vorkommen (z.B. Medication), werden nur einmal gezählt.

**Zeitpunkt.** Jeder Lauf ist eine Momentaufnahme zum angegebenen Datum. Liegen Läufe weit auseinander, können Unterschiede auch durch zwischenzeitlich geladene Daten entstehen.

**Programmversionen.** Ab Version 2026.11 erfasst FHIRScan zusätzlich die Verteilung über Kalenderjahre. Alle übrigen Auswertungen sind mit älteren Läufen voll vergleichbar. Bei der Gültigkeit von Verweisen werden ältere Läufe so ausgewertet wie neue (Kapitel 4).

# Anhang: Technische Details

## Uneinheitliche Datentypen

In [ ]:
#| output: asis
TI = F[F.type_inconsistency].copy()
if not TI.empty:
    # Listen sind kein eigener Typ; Ganzzahl und Dezimalzahl gemischt ist in JSON normal (5.0 wird als 5 geschrieben)
    typen = TI.detected_python_types.map(lambda t: {x.strip() for x in str(t).split(",")} - {"array"})
    TI = TI[typen.map(lambda t: len(t) > 1 and not t <= {"integer", "float"})]
    TI["detected_python_types"] = typen[TI.index].map(lambda t: ", ".join(sorted(t)))
if TI.empty:
    print("Bei keinem DIZ kommen Felder mit uneinheitlichem Datentyp vor. Zahlenfelder, die teils ganze Zahlen und teils "
          "Dezimalzahlen enthalten, sind dabei nicht berücksichtigt: In JSON wird z.B. 5,0 als 5 geschrieben, das ist unproblematisch.\n")
else:
    out = TI.pivot_table(index=["resource_type", "field_path"], columns="diz", values="detected_python_types", aggfunc="first")
    out = out.reindex(columns=DIZ_NAMEN).fillna("einheitlich oder nicht vorhanden").reset_index()
    out["field_path"] = [kurz(p, rt) for p, rt in zip(out.field_path, out.resource_type)]
    tabelle(out.rename(columns={"resource_type": "Ressourcentyp", "field_path": "Feld"}))
    quelle("*_fields.csv, Spalten type_inconsistency und detected_python_types")
    erklaerung(
        "Felder, deren Wert nicht in allen Ressourcen denselben Datentyp hat, z.B. mal Text und mal Zahl.",
        "Solche Felder erschweren die automatische Verarbeitung. Listen (array) zählen nicht als eigener Typ, und die Mischung "
        "aus ganzen Zahlen und Dezimalzahlen ist ausgenommen, weil JSON z.B. 5,0 als 5 schreibt.",
        "Exakte Erfassung über alle Ressourcen.")

## Verschachtelungstiefe

In [ ]:
#| output: asis
tiefe = pd.DataFrame({d: {rt: v.get("mean_depth") for rt, v in STRUKTUR[d].items()} for d in DIZ_NAMEN}).reindex(columns=DIZ_NAMEN)
if tiefe.empty:
    print("Keine Daten vorhanden.\n")
else:
    out = tiefe.sort_index().map(lambda v: "n.v." if pd.isna(v) else dez(float(v), 1)).reset_index().rename(columns={"index": "Ressourcentyp"})
    tabelle(out)
    quelle("07_structural_analysis.txt, mean_depth")
    erklaerung("Mittlere Verschachtelungstiefe der Ressourcen, also wie viele Ebenen ineinander liegender Angaben es gibt.",
               "Deutlich unterschiedliche Werte deuten auf unterschiedlich aufgebaute Ressourcen hin, z.B. durch zusätzliche Erweiterungen.",
               "Exakte Erfassung; für den inhaltlichen Vergleich von geringer Bedeutung.")

In [ ]:
#| include: false
# ── Excel-Tabellen: gleiche Inhalte, vollständig und filterbar ──────────────
# Wird nur erzeugt, wenn export_report.sh den Zielordner setzt.
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.styles.table import TableStyle, TableStyleElement, TableStyleList
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

ZIEL = os.environ.get("FHIRSCAN_OUT_DIR")
PROZENT = "0.00 %"


def bhc_tabellenformat(wb):
    linie = Side(style="thin", color="FFC9D4DA")
    ids = [wb._differential_styles.add(d) for d in (
        DifferentialStyle(border=Border(left=linie, right=linie, top=linie, bottom=linie, horizontal=linie, vertical=linie)),
        DifferentialStyle(font=Font(b=True, color="FFFFFFFF"), fill=PatternFill(patternType="solid", bgColor="FF1A3A5C")),
        DifferentialStyle(fill=PatternFill(patternType="solid", bgColor="FFF3F7FB")))]
    wb._table_styles = TableStyleList()
    wb._table_styles.tableStyle = [TableStyle(name="BHC", pivot=False, count=3, tableStyleElement=[
        TableStyleElement(type="wholeTable", dxfId=ids[0]), TableStyleElement(type="headerRow", dxfId=ids[1]),
        TableStyleElement(type="secondRowStripe", dxfId=ids[2])])]


def blatt(wb, name, titel, df, prozent=()):
    ws = wb.create_sheet(name)
    ws.sheet_view.showGridLines = False
    ausr = Alignment(horizontal="left", vertical="center", wrap_text=False)
    ws.cell(1, 1, titel).font = Font(name="Aptos", size=12, bold=True, color="1A3A5C")
    ws.cell(2, 1, f"Stand {STAND} · Version 1.0").font = Font(name="Aptos", size=9, color="4A5568")
    for j, c in enumerate(df.columns, 1):
        z = ws.cell(4, j, str(c))
        z.font = Font(name="Aptos", size=10, bold=True, color="FFFFFF")
        z.fill = PatternFill("solid", fgColor="1A3A5C")
        z.alignment = ausr
    for i, row in enumerate(df.itertuples(index=False), 5):
        for j, v in enumerate(row, 1):
            if isinstance(v, np.bool_):
                v = bool(v)
            elif isinstance(v, (np.integer,)):
                v = int(v)
            elif isinstance(v, (np.floating, float)):
                v = None if np.isnan(v) else float(v)
            if isinstance(v, float):
                # höchstens zwei Nachkommastellen (bei Anteilen: zwei Stellen im Prozentwert)
                v = round(v, 4) if df.columns[j - 1] in prozent else round(v, 2)
                if df.columns[j - 1] not in prozent and v.is_integer():
                    v = int(v)
            elif v is not None and not isinstance(v, (int, float, str)) and pd.isna(v):
                v = None
            z = ws.cell(i, j, v)
            z.font = Font(name="Aptos", size=10, color="262626")
            z.alignment = ausr
            if df.columns[j - 1] in prozent:
                z.number_format = PROZENT
            elif isinstance(v, int):
                z.number_format = "#,##0"
            elif isinstance(v, float):
                z.number_format = "#,##0.00"
    letzte = 4 + max(len(df), 1)
    t = Table(displayName="tbl_" + re.sub(r"\W", "_", name), ref=f"A4:{get_column_letter(len(df.columns))}{letzte}")
    t.tableStyleInfo = TableStyleInfo(name="BHC", showRowStripes=True)
    ws.add_table(t)
    ws.freeze_panes = "A5"
    for j, c in enumerate(df.columns, 1):
        breite = max([len(str(c))] + [len(str(v)) for v in df.iloc[:200, j - 1]]) + 2
        ws.column_dimensions[get_column_letter(j)].width = min(max(breite, 10), 70)
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True


if ZIEL:
    wb = Workbook()
    wb.remove(wb.active)
    bhc_tabellenformat(wb)
    steck = pd.DataFrame([{"DIZ": d, "Erhebung": LAUF[d]["modus"], "Lauf vom": LAUF[d]["datum"].strftime("%d.%m.%Y") if LAUF[d].get("datum") else None,
                           "Patienten": PATIENTEN[d], "Ressourcen": int(S[S.diz == d].total_resources.sum()),
                           "Felddaten": QUELLE[d], "Programmversion": LAUF[d].get("version", "vor 2026.11")} for d in DIZ_NAMEN])
    blatt(wb, "DIZ", "DIZ | Datengrundlage je DIZ", steck)
    res = S[["resource_type", "diz", "total_resources"]].copy()
    res["je_patient"] = res.total_resources / res.diz.map(PATIENTEN)
    blatt(wb, "Ressourcen", "Ressourcen | Anzahl je Ressourcentyp und DIZ",
          res.rename(columns={"resource_type": "Ressourcentyp", "diz": "DIZ", "total_resources": "Ressourcen", "je_patient": "Je Patient"}))
    fe = PIV.copy()
    fe.columns = [f"Befüllt {d}" for d in fe.columns]
    fe = fe.reset_index().merge(KL, on=["resource_type", "field_path"], how="left")
    fe = fe[["resource_type", "field_path", "klasse", "spanne"] + [f"Befüllt {d}" for d in DIZ_NAMEN]]
    blatt(wb, "Felder", "Felder | Befüllungsgrad je DIZ",
          fe.rename(columns={"resource_type": "Ressourcentyp", "field_path": "Feld", "klasse": "Klasse", "spanne": "Spanne"}),
          prozent=["Spanne"] + [f"Befüllt {d}" for d in DIZ_NAMEN])
    if not Q.empty:
        blatt(wb, "Verknüpfung", "Verknüpfung | Bezug zu Patient und Encounter",
              Q[["resource_type", "diz", "total_resources", "patient_linkage_rate", "encounter_linkage_rate"]].rename(columns={
                  "resource_type": "Ressourcentyp", "diz": "DIZ", "total_resources": "Ressourcen",
                  "patient_linkage_rate": "Mit Patientenbezug", "encounter_linkage_rate": "Mit Encounter-Bezug"}),
              prozent=["Mit Patientenbezug", "Mit Encounter-Bezug"])
    if not R.empty:
        blatt(wb, "Verweise", "Verweise | Gültigkeit je Verweisart",
              R.rename(columns={"diz": "DIZ", "pfad": "Verweis von", "ziel": "Ziel", "gesamt": "Verweise", "gueltig": "Gefunden",
                                "nicht_gefunden": "Nicht gefunden", "status": "Status", "anteil_gueltig": "Anteil gefunden"}),
              prozent=["Anteil gefunden"])
    if not C.empty:
        blatt(wb, "Je Patient", "Verteilung | Ressourcen je Patient und Encounter",
              C[["resource_type", "anchor_type", "diz", "coverage_rate", "min", "p25", "median", "p75", "p90", "p99", "max", "mean"]].rename(columns={
                  "resource_type": "Ressourcentyp", "anchor_type": "Bezug", "diz": "DIZ", "coverage_rate": "Anteil mit Ressource",
                  "median": "Median", "mean": "Mittelwert", "min": "Minimum", "max": "Maximum", "p25": "P25", "p75": "P75", "p90": "P90", "p99": "P99"}),
              prozent=["Anteil mit Ressource"])
    werte = [{"Ressourcentyp": r.resource_type, "Feld": r.field_path, "DIZ": r.diz, "Rang": i, "Wert": w, "Anzahl": n,
              "Anteil": n / werte_nenner(r) if werte_nenner(r) else None}
             for r in F[F.top_values.notna()].itertuples() for i, (w, n) in enumerate(top_werte(r.top_values), 1)]
    if werte:
        blatt(wb, "Werte", "Werte | Häufigste Werte je Feld", pd.DataFrame(werte), prozent=["Anteil"])
    if not T.empty:
        blatt(wb, "Zeitraum", "Zeitraum | Früheste und späteste Angabe",
              T[["resource_type", "diz", "anzahl", "frueh", "spaet", "unplausibel"]].rename(columns={
                  "resource_type": "Ressourcentyp", "diz": "DIZ", "anzahl": "Datierte Ressourcen", "frueh": "Früheste Angabe",
                  "spaet": "Späteste Angabe", "unplausibel": "Unplausibel"}).assign(Unplausibel=lambda x: x.Unplausibel.map({True: "ja", False: "nein"})))
    if not J.empty:
        blatt(wb, "Jahre", "Zeitraum | Ressourcen je Kalenderjahr",
              J.rename(columns={"diz": "DIZ", "resource_type": "Ressourcentyp", "jahr": "Jahr", "anzahl": "Ressourcen"}))
    if not PR.empty:
        blatt(wb, "Profile", "Profile | Verwendete Profile und Versionen",
              PR[["resource_type", "profil", "bereich", "version", "diz", "anzahl", "anteil", "url"]].rename(columns={
                  "resource_type": "Ressourcentyp", "profil": "Profil", "bereich": "Bereich", "version": "Version", "diz": "DIZ",
                  "anzahl": "Ressourcen", "anteil": "Anteil", "url": "Profil-URL"}), prozent=["Anteil"])
    stamp = os.environ.get("FHIRSCAN_STAMP", datetime.now().strftime("%Y%m%d_%H%M"))
    wb.save(Path(ZIEL) / f"FHIRScan_DIZ_Tabellen_{stamp}.xlsx")